In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.feature_selection import SelectKBest, f_regression, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import classification_report, mean_squared_error, r2_score
import optuna
from hyperopt import fmin, tpe, hp, Trials, space_eval
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
from ray import train, tune

/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-05-07 20:00:24,517	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-05-07 20:00:24,665	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


# Импорт и подготовка данных

In [2]:
reg_df = pd.read_csv("../data/Regression_wine_quality_filtered.csv", sep=",", encoding="utf-8", index_col=0)
clf_df = pd.read_csv("../data/Classification_smoke_detectors_filtered.csv", sep=",", encoding="utf-8", index_col=0).head(10000)

In [3]:
X_reg = reg_df.drop('quality', axis=1)
y_reg = reg_df['quality']

X_clf = clf_df.drop('Fire Alarm', axis=1)
y_clf = clf_df['Fire Alarm']

In [4]:
scaler_reg = StandardScaler()
X_reg_scaled = scaler_reg.fit_transform(X_reg)

scaler_clf = StandardScaler()
X_clf_scaled = scaler_clf.fit_transform(X_clf)

In [5]:
X_reg_sel = SelectKBest(f_regression, k=5).fit_transform(X_reg_scaled, y_reg)
X_clf_sel = SelectKBest(f_classif, k=2).fit_transform(X_clf_scaled, y_clf)

In [6]:
kf_reg = KFold(n_splits=5, shuffle=True, random_state=42)
kf_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [7]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg_sel, y_reg, test_size=0.2, random_state=42
)

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf_sel, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# Sklearn MLP

## Регрессия

### Optuna

In [8]:
def optuna_reg_obj(trial):
    hls_choice = trial.suggest_categorical('hidden_layer_sizes', ['32', '64'])
    hls = (int(hls_choice),)

    p = {
        'hidden_layer_sizes': hls,
        'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
        'solver': trial.suggest_categorical('solver', ['adam', 'sgd', 'lbfgs']),
        'alpha': trial.suggest_float('alpha', 1e-4, 1e-2, log=True),
        'max_iter': 40000,
        'random_state': 42
    }
    return cross_val_score(MLPRegressor(**p), X_reg_scaled, y_reg, cv=kf_reg, scoring='r2').mean()

In [9]:
reg_optuna = optuna.create_study(direction='maximize')
reg_optuna.optimize(optuna_reg_obj, n_trials=5)
best_params_reg_optuna = reg_optuna.best_params.copy()
best_params_reg_optuna['hidden_layer_sizes'] = (int(best_params_reg_optuna['hidden_layer_sizes']),)

[I 2026-05-07 20:00:24,996] A new study created in memory with name: no-name-e25557df-22eb-478b-9bd0-fd47193abb73
[I 2026-05-07 20:00:43,488] Trial 0 finished with value: -0.0642248305057199 and parameters: {'hidden_layer_sizes': '32', 'activation': 'tanh', 'solver': 'lbfgs', 'alpha': 0.0010438765645930357}. Best is trial 0 with value: -0.0642248305057199.
[I 2026-05-07 20:01:10,363] Trial 1 finished with value: -0.47483758911666624 and parameters: {'hidden_layer_sizes': '64', 'activation': 'relu', 'solver': 'lbfgs', 'alpha': 0.0015051382257961338}. Best is trial 0 with value: -0.0642248305057199.
[I 2026-05-07 20:01:18,364] Trial 2 finished with value: 0.36477457365484023 and parameters: {'hidden_layer_sizes': '64', 'activation': 'tanh', 'solver': 'adam', 'alpha': 0.0022028413123640483}. Best is trial 2 with value: 0.36477457365484023.
[I 2026-05-07 20:01:20,141] Trial 3 finished with value: 0.3502220475472536 and parameters: {'hidden_layer_sizes': '32', 'activation': 'relu', 'solver'

In [10]:
model = MLPRegressor(**best_params_reg_optuna)
model.fit(X_train_reg, y_train_reg)

,"loss loss: {'squared_error', 'poisson'}, default='squared_error'The loss function to use when training the weights. Note that the""squared error"" and ""poisson"" losses actually implement""half squares error"" and ""half poisson deviance"" to simplify thecomputation of the gradient. Furthermore, the ""poisson"" loss internally usesa log-link (exponential as the output activation function) and requires``y >= 0``... versionchanged:: 1.7 Added parameter `loss` and option 'poisson'.",'squared_error'
,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(64,)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'tanh'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.",0.0022028413123640483
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the regressor will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate ``learning_rate_`` at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when solver='sgd'.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


In [11]:
y_pred = model.predict(X_test_reg)
print(r2_score(y_test_reg, y_pred), mean_squared_error(y_test_reg, y_pred))

0.42848502615880835 0.3734883841257686


### RandomizedSearchCV

In [12]:
reg_rs = RandomizedSearchCV(
    MLPRegressor(max_iter=40000, random_state=42),
    {
        'hidden_layer_sizes': [(32,), (64,)],
        'activation': ['relu', 'tanh'],
        'solver': ['adam', 'sgd', 'lbfgs'],
        'alpha': np.logspace(-4, -2, 3)
    },
    n_iter=3, cv=kf_reg, scoring='r2', random_state=42, n_jobs=1
)

In [13]:
reg_rs.fit(X_train_reg, y_train_reg)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",MLPRegressor(...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'activation': ['relu', 'tanh'], 'alpha': array([0.0001...001 , 0.01 ]), 'hidden_layer_sizes': [(32,), (64,)], 'solver': ['adam', 'sgd', ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",3
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used h

In [14]:
y_pred = reg_rs.predict(X_test_reg)
print(r2_score(y_test_reg, y_pred), mean_squared_error(y_test_reg, y_pred))

0.41379023163741724 0.38309151844780936


### Hyperopt

In [15]:
def hyperopt_reg_obj(p):
    p['max_iter'] = 40000
    p['random_state'] = 42
    return -cross_val_score(MLPRegressor(**p), X_reg_scaled, y_reg, cv=kf_reg, scoring='r2').mean()

In [16]:
hyperopt_reg_space = {
    'hidden_layer_sizes': hp.choice('hls', [(32,), (64,)]),
    'activation': hp.choice('act', ['relu', 'tanh']),
    'solver': hp.choice('sol', ['adam', 'sgd', 'lbfgs']),
    'alpha': hp.loguniform('alpha', np.log(1e-4), np.log(1e-2))
}

In [17]:
reg_hyperopt = fmin(hyperopt_reg_obj, space=hyperopt_reg_space, algo=tpe.suggest, max_evals=5, trials=Trials())
best_params_hyperopt_reg = space_eval(hyperopt_reg_space, reg_hyperopt)

100%|██████████| 5/5 [00:53<00:00, 10.71s/trial, best loss: -0.35023387966552544]


In [18]:
model = MLPRegressor(**best_params_hyperopt_reg)
model.fit(X_train_reg, y_train_reg)

,"loss loss: {'squared_error', 'poisson'}, default='squared_error'The loss function to use when training the weights. Note that the""squared error"" and ""poisson"" losses actually implement""half squares error"" and ""half poisson deviance"" to simplify thecomputation of the gradient. Furthermore, the ""poisson"" loss internally usesa log-link (exponential as the output activation function) and requires``y >= 0``... versionchanged:: 1.7 Added parameter `loss` and option 'poisson'.",'squared_error'
,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(64,)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'tanh'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'sgd'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.",0.0012782232121115282
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the regressor will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate ``learning_rate_`` at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when solver='sgd'.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


In [19]:
y_pred = model.predict(X_test_reg)
print(r2_score(y_test_reg, y_pred), mean_squared_error(y_test_reg, y_pred))

0.38294971091410435 0.4032459794466704


## Классификация

### Optuna

In [20]:
def optuna_clf_obj(trial):
    hls_choice = trial.suggest_categorical('hidden_layer_sizes', ['32', '64'])
    hls = (int(hls_choice),)

    p = {
        'hidden_layer_sizes': hls,
        'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
        'solver': trial.suggest_categorical('solver', ['adam', 'sgd', 'lbfgs']),
        'alpha': trial.suggest_float('alpha', 1e-4, 1e-2, log=True),
        'max_iter': 40000,
        'random_state': 42
    }
    return cross_val_score(MLPClassifier(**p), X_clf_scaled, y_clf, cv=kf_clf, scoring='accuracy').mean()

clf_optuna = optuna.create_study(direction='maximize')
clf_optuna.optimize(optuna_clf_obj, n_trials=3)
best_params_clf_optuna = clf_optuna.best_params.copy()
best_params_clf_optuna['hidden_layer_sizes'] = (int(best_params_clf_optuna['hidden_layer_sizes']),)

In [22]:
clf_optuna = optuna.create_study(direction='maximize')
clf_optuna.optimize(optuna_clf_obj, n_trials=3)
best_params_clf_optuna = clf_optuna.best_params.copy()
best_params_clf_optuna['hidden_layer_sizes'] = (int(best_params_clf_optuna['hidden_layer_sizes']),)

[I 2026-05-07 20:04:56,357] A new study created in memory with name: no-name-fddd57d0-8fae-4923-83d9-bffb3cdea86e
[I 2026-05-07 20:04:58,135] Trial 0 finished with value: 0.999 and parameters: {'hidden_layer_sizes': '32', 'activation': 'tanh', 'solver': 'adam', 'alpha': 0.00010361087039970448}. Best is trial 0 with value: 0.999.
[I 2026-05-07 20:04:58,441] Trial 1 finished with value: 0.9994000000000002 and parameters: {'hidden_layer_sizes': '32', 'activation': 'relu', 'solver': 'lbfgs', 'alpha': 0.0004458838963995265}. Best is trial 1 with value: 0.9994000000000002.
[I 2026-05-07 20:05:00,957] Trial 2 finished with value: 0.9994 and parameters: {'hidden_layer_sizes': '64', 'activation': 'relu', 'solver': 'adam', 'alpha': 0.00014597992129537174}. Best is trial 1 with value: 0.9994000000000002.


In [23]:
model = MLPClassifier(**best_params_clf_optuna)
model.fit(X_train_clf, y_train_clf)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(32,)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'lbfgs'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0004458838963995265
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",None


In [24]:
y_pred = model.predict(X_test_clf)
print(classification_report(y_test_clf, y_pred))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       636
         1.0       1.00      1.00      1.00      1364

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



### RandomizedSearchCV

In [25]:
clf_rs = RandomizedSearchCV(
    MLPClassifier(max_iter=40000, random_state=42),
    {
        'hidden_layer_sizes': [(32,), (64,)],
        'activation': ['relu', 'tanh'],
        'solver': ['adam', 'sgd', 'lbfgs'],
        'alpha': np.logspace(-4, -2, 3)
    },
    n_iter=3, cv=kf_clf, scoring='accuracy', random_state=42, n_jobs=1
)

In [26]:
clf_rs.fit(X_train_clf, y_train_clf)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",MLPClassifier...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'activation': ['relu', 'tanh'], 'alpha': array([0.0001...001 , 0.01 ]), 'hidden_layer_sizes': [(32,), (64,)], 'solver': ['adam', 'sgd', ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",3
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be 

In [27]:
y_pred = clf_rs.predict(X_test_clf)
print(classification_report(y_test_clf, y_pred))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       636
         1.0       1.00      1.00      1.00      1364

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



### Hyperopt

In [28]:
def hyperopt_clf_obj(p):
    p['max_iter'] = 40000
    p['random_state'] = 42
    return -cross_val_score(MLPClassifier(**p), X_clf_scaled, y_clf, cv=kf_clf, scoring='accuracy').mean()

hyperopt_clf_space = {
    'hidden_layer_sizes': hp.choice('hls', [(32,), (64,)]),
    'activation': hp.choice('act', ['relu', 'tanh']),
    'solver': hp.choice('sol', ['adam', 'sgd', 'lbfgs']),
    'alpha': hp.loguniform('alpha', np.log(1e-4), np.log(1e-2))
}

In [29]:
best_hyperopt = fmin(hyperopt_clf_obj, space=hyperopt_clf_space, algo=tpe.suggest, max_evals=3, trials=Trials())
best_params_hyperopt = space_eval(hyperopt_clf_space, best_hyperopt)

100%|██████████| 3/3 [00:08<00:00,  2.91s/trial, best loss: -0.999]


In [30]:
model = MLPClassifier(**best_params_hyperopt)
model.fit(X_train_clf, y_train_clf)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(32,)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'tanh'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0015957023270261723
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",None


In [31]:
y_pred = model.predict(X_test_clf)
print(classification_report(y_test_clf, y_pred))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       636
         1.0       1.00      1.00      1.00      1364

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



# Tensorflow

## Регрессия

### Optuna

In [32]:
def optuna_tf_reg(trial):
    units = trial.suggest_int('units', 64, 256, step=64)
    opt = trial.suggest_categorical('optimizer', ['adam', 'sgd', 'rmsprop'])
    batch = trial.suggest_int('batch_size', 64, 512, step=64)
    epochs = 30
    
    model = keras.Sequential([
        layers.Dense(units, activation='relu', input_shape=(X_reg_sel.shape[1],)), 
        layers.Dense(1)
    ])
    
    model.compile(optimizer=opt, loss='mse')
    
    model.fit(X_reg_sel, y_reg, epochs=epochs, batch_size=batch, 
              validation_split=0.2, verbose=0)
    
    y_pred = model.predict(X_reg_sel, verbose=0).flatten()
    ss_res = np.sum((y_reg - y_pred) ** 2)
    ss_tot = np.sum((y_reg - np.mean(y_reg)) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    
    return r2 

In [33]:
study = optuna.create_study(direction='maximize')
study.optimize(optuna_tf_reg, n_trials=5, show_progress_bar=True)

[I 2026-05-07 20:05:32,257] A new study created in memory with name: no-name-fb7366e1-a6bb-4e40-8edf-483f191f64cf
  0%|          | 0/5 [00:00<?, ?it/s]/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-05-07 20:05:32.269752: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2026-05-07 20:05:32.269843: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
2026-05-07 20:05:32.269858: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.66 GB
2026-05-07 20:05:32.269888: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your 

[I 2026-05-07 20:05:35,375] Trial 0 finished with value: -3.1682648885590563 and parameters: {'units': 128, 'optimizer': 'adam', 'batch_size': 256}. Best is trial 0 with value: -3.1682648885590563.


Best trial: 1. Best value: -3.05909:  40%|████      | 2/5 [00:04<00:06,  2.26s/it]

[I 2026-05-07 20:05:37,037] Trial 1 finished with value: -3.0590854323895167 and parameters: {'units': 128, 'optimizer': 'rmsprop', 'batch_size': 256}. Best is trial 1 with value: -3.0590854323895167.


Best trial: 1. Best value: -3.05909:  60%|██████    | 3/5 [00:06<00:04,  2.13s/it]

[I 2026-05-07 20:05:39,007] Trial 2 finished with value: -9.354979816507702 and parameters: {'units': 192, 'optimizer': 'rmsprop', 'batch_size': 448}. Best is trial 1 with value: -3.0590854323895167.


Best trial: 3. Best value: 0.337039:  80%|████████  | 4/5 [00:10<00:02,  2.78s/it]

[I 2026-05-07 20:05:42,778] Trial 3 finished with value: 0.3370389153532003 and parameters: {'units': 256, 'optimizer': 'rmsprop', 'batch_size': 64}. Best is trial 3 with value: 0.3370389153532003.


Best trial: 4. Best value: 0.340049: 100%|██████████| 5/5 [00:12<00:00,  2.46s/it]

[I 2026-05-07 20:05:44,558] Trial 4 finished with value: 0.34004925765245897 and parameters: {'units': 64, 'optimizer': 'sgd', 'batch_size': 384}. Best is trial 4 with value: 0.34004925765245897.


In [34]:
best_params_tf_reg = study.best_params
print("Лучшие гиперпараметры:", best_params_tf_reg)

Лучшие гиперпараметры: {'units': 64, 'optimizer': 'sgd', 'batch_size': 384}


In [35]:
def build_and_train_final_reg_model(params, X, y):
    model = keras.Sequential([
        layers.Dense(params['units'], activation='relu', input_shape=(X.shape[1],)),
        layers.Dense(1)
    ])
    
    model.compile(optimizer=params['optimizer'], loss='mse')
    
    model.fit(
        X, y, 
        epochs=100, 
        batch_size=params['batch_size'],
        verbose=0
    )
    return model

In [36]:
optuna_model = build_and_train_final_reg_model(best_params_tf_reg, X_train_reg, y_train_reg)

In [37]:
y_pred = optuna_model.predict(X_test_reg)
print(r2_score(y_test_reg, y_pred))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
0.3892430067062378


### KerasTuner

In [38]:
def kt_reg_builder(hp):
    model = keras.Sequential([
        keras.layers.Dense(
            hp.Int('units', 64, 256, step=64), 
            activation='relu', 
            input_shape=(X_reg_sel.shape[1],)
        ), 
        layers.Dense(1)
    ])
    
    model.compile(
        optimizer=hp.Choice('optimizer', ['adam', 'sgd', 'rmsprop']), 
        loss='mse'
    )
    return model

In [39]:
tuner = kt.RandomSearch(
    kt_reg_builder, 
    objective='val_loss', 
    max_trials=2, 
    directory='kt_reg', 
    project_name='reg'
)

Reloading Tuner from kt_reg/reg/tuner0.json


In [40]:
tuner.search(X_reg_sel, y_reg, epochs=1, validation_split=0.2, verbose=0)
best_model = tuner.get_best_models()[0]
best_model.fit(X_train_reg, y_train_reg, epochs=10, verbose=0)

/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [41]:
y_pred = best_model.predict(X_test_reg)
print(r2_score(y_test_reg, y_pred))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
0.37996572256088257


### Ray Tune

In [42]:
def train_model_reg(config):
    X_reg_train = config["X_data"]
    y_reg_train = config["y_data"]
    
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(config["units"], activation="relu"),
        tf.keras.layers.Dense(1, activation="linear")
    ])
    
    optimizers = {
        "adam": tf.keras.optimizers.Adam, 
        "sgd": tf.keras.optimizers.SGD, 
        "rmsprop": tf.keras.optimizers.RMSprop
    }
    
    model.compile(
        optimizer=optimizers[config["optimizer"]](learning_rate=config["lr"]), 
        loss="mean_squared_error", 
        metrics=["mae"]
    )
    
    model.fit(
        X_reg_train, y_reg_train, 
        epochs=config["epochs"], 
        batch_size=config["batch_size"], 
        verbose=0,
        validation_split=0.2
    )
    
    loss, mae = model.evaluate(X_reg_train, y_reg_train, verbose=0)
    tune.report({"loss": loss, "mae": mae})

In [43]:
param_space_reg = {
    "optimizer": tune.choice(["adam", "sgd", "rmsprop"]),
    "lr": tune.loguniform(1e-4, 1e-2),
    "units": tune.choice([16, 32, 64]),
    "batch_size": tune.choice([16, 32]),
    "epochs": 20,
    "X_data": X_train_reg,
    "y_data": y_train_reg
}

In [44]:
tuner_reg = tune.Tuner(
    train_model_reg,
    param_space=param_space_reg,
    tune_config=tune.TuneConfig(num_samples=6)
)

In [45]:
best_reg = tuner_reg.fit().get_best_result(metric="loss", mode="min")
best_config_reg = best_reg.config

(train_model_reg pid=8810) 2026-05-07 20:05:59.606040: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
(train_model_reg pid=8810) 2026-05-07 20:05:59.606066: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
(train_model_reg pid=8810) 2026-05-07 20:05:59.606081: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.66 GB
(train_model_reg pid=8810) 2026-05-07 20:05:59.606097: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
(train_model_reg pid=8810) 2026-05-07 20:05:59.606108: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
(train_model_reg pid=8810) 2026-05-07 20:05:59.

In [46]:
final_model = tf.keras.Sequential([
    tf.keras.layers.Dense(best_config_reg["units"], activation="relu"),
    tf.keras.layers.Dense(1, activation="linear")
])

In [47]:
optimizers = {
    "adam": tf.keras.optimizers.Adam, 
    "sgd": tf.keras.optimizers.SGD, 
    "rmsprop": tf.keras.optimizers.RMSprop
}

In [48]:
final_model.compile(
    optimizer=optimizers[best_config_reg["optimizer"]](learning_rate=best_config_reg["lr"]),
    loss="mean_squared_error",
    metrics=["mae"]
)


In [49]:
history = final_model.fit(
    X_train_reg, y_train_reg,
    epochs=100,
    batch_size=best_config_reg["batch_size"],
    validation_split=0.2,
    verbose=0
)

In [50]:
y_pred = final_model.predict(X_test_reg)
print(r2_score(y_test_reg, y_pred))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
0.38638001680374146


## Классификация

### Optuna

In [51]:
def optuna_tf_clf(trial):
    units = trial.suggest_int('units', 64, 256, step=64)
    opt = trial.suggest_categorical('optimizer', ['adam', 'sgd', 'rmsprop'])
    batch = trial.suggest_int('batch_size', 64, 512, step=64)
    epochs = 30
    
    model = keras.Sequential([
        layers.Dense(units, activation='relu', input_shape=(X_clf_sel.shape[1],)), 
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    
    model.fit(X_clf_sel, y_clf, epochs=epochs, batch_size=batch, 
              validation_split=0.2, verbose=0)
    
    y_pred_prob = model.predict(X_clf_sel, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int).ravel()
    report = classification_report(y_clf.values.ravel(), y_pred, output_dict=True)
    return report['accuracy']

In [52]:
study = optuna.create_study(direction='maximize')
study.optimize(optuna_tf_clf, n_trials=5, show_progress_bar=True)

[I 2026-05-07 20:06:27,787] A new study created in memory with name: no-name-f277b585-994b-46cc-8c1b-ba2f8a0df368
  0%|          | 0/5 [00:00<?, ?it/s]/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
Best trial: 0. Best value: 0.9904:  20%|██        | 1/5 [00:06<00:25,  6.27s/it]/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[I 2026-05-07 20:06:34,061] Trial 0 finished with value: 0.9904 and parameters: {'units': 192, 'optimizer': 'sgd', 'batch_size': 256}. Best is trial 0 with value: 0.9904.


Best trial: 1. Best value: 0.9993:  40%|████      | 2/5 [00:14<00:21,  7.20s/it]/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[I 2026-05-07 20:06:41,907] Trial 1 finished with value: 0.9993 and parameters: {'units': 128, 'optimizer': 'rmsprop', 'batch_size': 192}. Best is trial 1 with value: 0.9993.


Best trial: 1. Best value: 0.9993:  60%|██████    | 3/5 [00:18<00:11,  5.88s/it]/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[I 2026-05-07 20:06:46,225] Trial 2 finished with value: 0.9979 and parameters: {'units': 256, 'optimizer': 'rmsprop', 'batch_size': 512}. Best is trial 1 with value: 0.9993.


Best trial: 1. Best value: 0.9993:  80%|████████  | 4/5 [00:24<00:05,  5.80s/it]/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[I 2026-05-07 20:06:51,895] Trial 3 finished with value: 0.989 and parameters: {'units': 256, 'optimizer': 'sgd', 'batch_size': 256}. Best is trial 1 with value: 0.9993.


Best trial: 1. Best value: 0.9993: 100%|██████████| 5/5 [00:37<00:00,  7.56s/it]

[I 2026-05-07 20:07:05,609] Trial 4 finished with value: 0.997 and parameters: {'units': 128, 'optimizer': 'sgd', 'batch_size': 64}. Best is trial 1 with value: 0.9993.


In [53]:
best_params_tf_clf = study.best_params
print("Лучшие гиперпараметры:", best_params_tf_clf)

Лучшие гиперпараметры: {'units': 128, 'optimizer': 'rmsprop', 'batch_size': 192}


In [54]:
def build_and_train_final_clf_model(params, X, y):
    model = keras.Sequential([
        layers.Dense(params['units'], activation='relu', input_shape=(X.shape[1],)),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer=params['optimizer'], loss='binary_crossentropy', metrics=['accuracy'])
    
    model.fit(
        X, y, 
        epochs=100, 
        batch_size=params['batch_size'],
        verbose=0
    )
    return model

In [55]:
optuna_model = build_and_train_final_clf_model(best_params_tf_clf, X_train_clf, y_train_clf)

/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [56]:
y_pred_prob = optuna_model.predict(X_test_clf)
y_pred = (y_pred_prob > 0.5).astype(int).ravel()
print(classification_report(y_test_clf, y_pred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       636
         1.0       1.00      1.00      1.00      1364

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



### KerasTuner

In [57]:
def kt_clf_builder(hp):
    model = keras.Sequential([
        keras.layers.Dense(
            hp.Int('units', 64, 256, step=64), 
            activation='relu', 
            input_shape=(X_clf_sel.shape[1],)
        ), 
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=hp.Choice('optimizer', ['adam', 'sgd', 'rmsprop']), 
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [58]:
tuner = kt.RandomSearch(
    kt_clf_builder, 
    objective='val_loss',
    max_trials=2, 
    directory='kt_clf', 
    project_name='clf'
)

Reloading Tuner from kt_clf/clf/tuner0.json


In [59]:
tuner.search(X_clf_sel, y_clf, epochs=1, validation_split=0.2, verbose=0)
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = tuner.hypermodel.build(best_hps)

/Users/slava/Development/Unik/ML/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [60]:
best_model.fit(X_train_clf, y_train_clf, epochs=10, validation_data=(X_test_clf, y_test_clf), verbose=0)

In [61]:
y_pred_proba = best_model.predict(X_test_clf, verbose=0)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()
print(classification_report(y_test_clf, y_pred))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       636
         1.0       1.00      1.00      1.00      1364

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



### Ray Tune

In [62]:
def train_model_clf(config):
    X_clf_train = config["X_data"]
    y_clf_train = config["y_data"]
    
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(config["units"], activation="relu"),
        tf.keras.layers.Dense(config["num_classes"], activation="softmax")
    ])
    
    optimizers = {
        "adam": tf.keras.optimizers.Adam, 
        "sgd": tf.keras.optimizers.SGD, 
        "rmsprop": tf.keras.optimizers.RMSprop
    }
    
    model.compile(
        optimizer=optimizers[config["optimizer"]](learning_rate=config["lr"]), 
        loss="sparse_categorical_crossentropy", 
        metrics=["accuracy"]
    )
    
    model.fit(
        X_clf_train, y_clf_train, 
        epochs=config["epochs"], 
        batch_size=config["batch_size"], 
        verbose=0,
        validation_split=0.2
    )
    
    loss, accuracy = model.evaluate(X_clf_train, y_clf_train, verbose=0)
    tune.report({"loss": loss, "accuracy": accuracy})

In [63]:
param_space_clf = {
    "optimizer": tune.choice(["adam", "sgd", "rmsprop"]),
    "lr": tune.loguniform(1e-4, 1e-2),
    "units": tune.choice([16, 32, 64]),
    "batch_size": tune.choice([16, 32]),
    "epochs": 20,
    "num_classes": len(np.unique(y_train_clf)),
    "X_data": X_train_clf,
    "y_data": y_train_clf
}

In [64]:
tuner_clf = tune.Tuner(
    train_model_clf,
    param_space=param_space_clf,
    tune_config=tune.TuneConfig(num_samples=6)
)

In [65]:
best_clf = tuner_clf.fit().get_best_result(metric="loss", mode="min")
best_config_clf = best_clf.config

(train_model_clf pid=8853) 2026-05-07 20:07:45.255837: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
(train_model_clf pid=8853) 2026-05-07 20:07:45.255881: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
(train_model_clf pid=8853) 2026-05-07 20:07:45.255914: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.66 GB
(train_model_clf pid=8853) 2026-05-07 20:07:45.255930: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
(train_model_clf pid=8853) 2026-05-07 20:07:45.255951: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
(train_model_clf pid=8851) 2026-05-07 20:07:45.

In [66]:
final_model = tf.keras.Sequential([
    tf.keras.layers.Dense(best_config_clf["units"], activation="relu"),
    tf.keras.layers.Dense(best_config_clf["num_classes"], activation="softmax")
])

In [67]:
optimizers = {
    "adam": tf.keras.optimizers.Adam, 
    "sgd": tf.keras.optimizers.SGD, 
    "rmsprop": tf.keras.optimizers.RMSprop
}

In [68]:
final_model.compile(
    optimizer=optimizers[best_config_clf["optimizer"]](learning_rate=best_config_clf["lr"]),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [69]:
history = final_model.fit(
    X_train_clf, y_train_clf,
    epochs=30,
    batch_size=best_config_clf["batch_size"],
    validation_split=0.2,
    verbose=0
)

In [70]:
y_pred = final_model.predict(X_test_clf)
y_pred_classes = np.argmax(y_pred, axis=1)
print(classification_report(y_test_clf, y_pred_classes))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       636
         1.0       1.00      1.00      1.00      1364

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

